# Lab 1 – K-Nearest Neighbours

In this lab session, we will implement the K-Nearest Neighbours (KNN) algorithm.  

Specifically, we will work through a typical machine learning pipeline, which involves:

1. Defining the problem we want to solve
2. Collecting the data
3. Preprocessing the data
4. Define the variables of the problem
5. Implementing a KNN model
6. Use the model to make predictions on unseen data

#### Lab Materials

 The KNN slides available on moodle.

# Step 1&2: Defining the Problem

In this lab, we will use the data collected during the course to create a model that predicts whether a student experiences continuous sleep (`continuous_sleep`) based on their daily habits (e.g., screen time, smoking habits, and sleep duration).

We begin by loading the dataset into a Pandas DataFrame.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv("Intro to ML Class data.csv")

In [ ]:
# First, we rename the columns for convenience.
rename_dict = {
    'Sleep Time': 'sleep_time',
    'Wake Time': 'wake_time',
    'Smoking (#e)': 'smoking',
    'Screen Time (hrs)': 'screen_time',
    'Conintuous Sleep?': 'continuous_sleep'}
data.rename(columns=rename_dict, inplace=True)

In [ ]:
data

Because the label takes two possible values, 'N' (No) and 'Y' (Yes), we can treat this as a binary classification problem.

For convenience, we relabel the two classes as 0 and 1, where 0 indicates non-continuous/interrupted sleep ('N') and 1 indicates continuous sleep ('Y').

In [ ]:
mapping = {
    'N': 0,
    'Y': 1
}

data['continuous_sleep'] = data['continuous_sleep'].map(mapping)

# Step 3: Data Preprocessing

As in many real-world applications, the raw data contains missing values (`NaN`), string-formatted numbers, and timestamps that require preprocessing before running machine learning algorithms.

There is no single correct way to preprocess a dataset, and the choices made during preprocessing can significantly affect the algorithm’s performance.

For example, we need to handle missing values, convert `screen_time` to continuous numeric values, and extract numerical feature representations like total `sleep_duration` from raw sleep and wake timestamps.

In [ ]:
# Clean up missing target values
data = data.dropna(subset=['continuous_sleep']).reset_index(drop=True)

# Option A: Drop rows where ANY of the specified columns have missing values
#feature_cols = ['sleep_time', 'wake_time', 'smoking', 'screen_time', 'continuous_sleep']
#data = data.dropna(subset=feature_cols).reset_index(drop=True)

# Option B: Drop rows where ALL columns have missing values
# data = data.dropna(how='all').reset_index(drop=True)


# Convert the column to float
data['screen_time'] = data['screen_time'].astype(float)

In [ ]:
data

In [ ]:
#  Convert columns to datetime first (just in case they are stored as strings)
data['sleep_time'] = pd.to_datetime(data['sleep_time'], format='%H:%M', errors='coerce')
data['wake_time'] = pd.to_datetime(data['wake_time'], format='%H:%M', errors='coerce')


#  Extract continuous numeric features (hours) for KNN distance metrics
data['sleep_time'] = data['sleep_time'].dt.hour + data['sleep_time'].dt.minute / 60.0
data['wake_time'] = data['wake_time'].dt.hour + data['wake_time'].dt.minute / 60.0

# Treat 00:xx as 24:xx
data.loc[data['sleep_time'] < 1, 'sleep_time'] += 24
data.loc[data['wake_time'] < 1, 'wake_time'] += 24

# Round to 2 decimals
data['sleep_time'] = data['sleep_time'].round(2)
data['wake_time'] = data['wake_time'].round(2)


In [ ]:
data

Before implementing KNN, examine the dataset and answer the following questions:

1. What problems or inconsistencies (e.g., scale differences, missing entries) do you observe in the raw data?
2. What preprocessing and feature transformation steps did you apply?
3. How might feature scaling (e.g., screen hours vs. smoking counts) affect distance calculations in the KNN algorithm?

# Step 4: Define the Problem Variables

Which variables will you use to predict `continuous_sleep`?

In [ ]:
# Define target label and feature variables
label = "continuous_sleep"

## TO DO
features = [] # fill this

X = data[features].values
y = data[label].values.astype(int)

# Step 5: Implement the KNN Algorithm

Below, we provide a template for the KNN model that you must complete.

More specifically, you should:

1. Implement the `get_probabilities(self, x)` method, which estimates the probability of each class for a test sample `x`.
2. Implement the `predict` method, which returns the class label with the highest estimated probability.

In [ ]:
def euclidean_metric(x, y):
    return np.linalg.norm(x - y)


## Skeleton code to be filled in
class NearestNeighbourClassifier:
    ## Initialise the neighbours with a specific metric function and dataset
    ## Assume labels are in {1, ..., m}
    def __init__(self, data, labels, metric, K):
        self.metric = metric
        self.data = data
        self.labels = labels
        self.n_classes = len(np.unique(labels))  # Counts actual number of labels
        self.K = K
        self.n_points = data.shape[0] # np array dimensions
        self.n_features = data.shape[1]
        print("classes: ", self.n_classes)

    def get_probabilities(self, x):
            # 1. Calculate distances from test point x to all training points
            # 2. Sort indices by distance using np.argsort()
            # 3. get K closest neighbours
            # 4. get the proportion of each label so that proportions[y] is the proportion of label y in the neighbourhood
            # 5. Return proportion
            pass

    ## predict the most likely label
    def predict(self, x):
        # Predict class label with highest probability
        # return using np.argmax
        pass

    # Gives a utility for every possible choice made by the algorithm
    def decide(self, U, x):
        """
        A method that return the action that maximise the expected utility.
        :param U: is a 2 denominational array that indicated the utility of each action a given y
                    example: U = np.array([ [ 1 , -1000],
                                            [ -1 ,    0]  ])
                            so U[1,0]=-1 is the utility of taking action a=1 when y=0.
        :param x: the test point.
        :return: the action that maximises the expected utility max_a E[U|a,x].
                 where E[U|a,x] = sum_y P(y|x) U(a,y).
        """
        # HINT:
        # Need to use the get_probabilities function to return the action with the highest
        # expected utility
        # i.e. maximising sum_y P(y|x) U(a,y)
        pass
    

In [ ]:
kNN = NearestNeighbourClassifier(data = , 
                                 labels= ,
                                 K = ,
                                 metric = euclidean_metric)

##### Question
After implementing your algorithm, evaluate its accuracy on the entire training set using \(k=1\) and \(k=5\).
What do you observe?

# Step 6: Utility-Based Decisions

We will now use a utility function to make predictions. 
More specifically, for each data point, we will select the action that maximizes the expected utility.

The utility matrix $U$ is a two-dimensional array in which $U[a,y]$ represents the utility of taking action $a$ when the true class is $y$.

For example:

```python
U = np.array([
    [ 1, -1000],
    [-1,     0]
])
```

In this example:

* $U[0,0]=1$: the utility of selecting action (or predict) $0$ when the true class is $0$.
* $U[0,1]=-1000$: the utility of selecting action $0$ when the true class is $1$.
* $U[1,0]=-1$: the utility of selecting action $1$ when the true class is $0$.
* $U[1,1]=0$: the utility of selecting action $1$ when the true class is $1$.


#### tasks
1) First, implement the `decide` method.
2) Then, use each of the two utility functions below (U_1 and U_2) to make predictions with KNN for \(k=3\), and calculate the resulting accuracy.

What do you observe? How does the accuracy obtained with each utility function compare with the accuracy of the standard KNN classifier?


In [ ]:
U_1 = np.array([[1, 0],
                [0, 1]])


U_2 = np.array([[1, -1000],
                [-1, 1]])